# Samuel Custom Voice Fine-Tune — Kaggle (GPU T4 x2, never TPU)

> **Accelerator:** `NvidiaTeslaT4` (2x T4 DDP)
> **Dataset:** `lydorandlydor/samuel-voice-samples`

**Repo is now PUBLIC** (`https://github.com/lydorianP/samuel-realtime-parrot`) — **no GH_TOKEN/HF_TOKEN secrets needed for clone.**
> Persistence: **Save Output ON** (all files in `/kaggle/working` persisted)


In [ ]:
import os, sys, subprocess, time, pathlib, shutil
print('='*60)
print('CELL 1: ENVIRONMENT & REPOSITORY SETUP')
print('='*60)

# Check for cached .venv + vendor from previous run (persistence)
cached_root = pathlib.Path('/kaggle/input/samuel-realtime-parrot-custom-train/samuel-realtime-parrot')
if cached_root.exists():
    print(f'Found cached repo at {cached_root}, restoring...')
    if not pathlib.Path('samuel-realtime-parrot').exists():
        shutil.copytree(cached_root, 'samuel-realtime-parrot', dirs_exist_ok=True)
        print('Restored repo + .venv + vendor from cache (skipping 120s install)')
    # Also restore uv cache
    cached_uv = pathlib.Path('/kaggle/input/samuel-realtime-parrot-custom-train/.cache')
    if cached_uv.exists():
        shutil.copytree(cached_uv, pathlib.Path.home() / '.cache', dirs_exist_ok=True)
    venv_ok = True
else:
    venv_ok = False

print('[1/5] Installing uv...')
subprocess.run('curl -LsSf https://astral.sh/uv/install.sh | sh', shell=True, check=True)
os.environ['PATH'] = '/root/.local/bin:' + os.environ['PATH']
print(subprocess.getoutput('uv --version'))

print('\n[2/5] Cloning public repository (no token needed)...')
if not pathlib.Path('samuel-realtime-parrot').exists():
    res = subprocess.run(['git', 'clone', 'https://github.com/lydorianP/samuel-realtime-parrot.git'], capture_output=True, text=True)
    if res.returncode != 0:
        print('[FATAL] Git clone failed')
        print(res.stderr)
        sys.exit(1)
    print('✅ Repository cloned.')
else:
    print('Repo exists (cached), pulling latest...')
    subprocess.run('cd samuel-realtime-parrot && git pull', shell=True)
os.chdir('samuel-realtime-parrot')
print(f'Working directory: {os.getcwd()}')

print('\n[3/5] Setting up Python 3.12...')
if venv_ok:
    out = subprocess.getoutput('.venv/bin/python -c "import hydra, torch; print(f\"Torch: {torch.__version__}\")"')
    if 'Torch:' in out:
        print(f'Cached .venv verified: {out.strip()}')
    else:
        print(f'Cached .venv check failed: {out[:200]}, reinstalling')
        venv_ok = False
        subprocess.run('uv python install 3.12', shell=True, check=True)
else:
    venv_ok = False
    subprocess.run('uv python install 3.12', shell=True, check=True)

if not venv_ok:
    print('\n[4/5] Syncing dependencies (with compatible PyTorch triad)...')
    # Install compatible PyTorch triad FIRST to avoid ABI mismatches
    subprocess.run('uv pip install torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 --index-url https://download.pytorch.org/whl/cu121', shell=True, check=True)
    subprocess.run('uv sync', shell=True, check=True)
    print('\n[5/5] Installing vendor/samuel (--no-deps to avoid torch conflicts)...')
    print('Initializing git submodules...')
    subprocess.run('git submodule update --init --recursive', shell=True, check=True)
    print('Installing vendor/samuel...')
    subprocess.run('pip install -e vendor/samuel --no-deps 2>&1 | tail -n 20', shell=True)
    # Install only the non-torch deps that vendor/samuel needs
    subprocess.run('pip install hydra-core omegaconf wandb transformers faster-whisper jiwer plotly tqdm julius 2>&1 | tail -n 10', shell=True)
else:
    print('Skipping [4/5] and [5/5] (using cached .venv with compatible triad)')
    # Still ensure submodules are present
    if not pathlib.Path('vendor/samuel/pyproject.toml').exists():
        subprocess.run('git submodule update --init --recursive', shell=True, check=True)

print('\n--- Verification ---')
print(subprocess.getoutput('python -c "import hydra; print(f\"hydra {hydra.__version__}\")" 2>&1 | head -n 5'))
print(subprocess.getoutput('python -c "import torch; print(f\"Torch: {torch.__version__}, CUDA: {torch.cuda.is_available()}, Devices: {torch.cuda.device_count()}\")" 2>&1 | head -n 5'))
print(subprocess.getoutput('python -c "import torchvision; print(f\"torchvision {torchvision.__version__}\")" 2>&1 | head -n 5'))
print(subprocess.getoutput('python -c "import torchaudio; print(f\"torchaudio {torchaudio.__version__}\")" 2>&1 | head -n 5'))
print('='*60)


In [ ]:
import pathlib, subprocess, numpy as np, json, os, time, sys
print('='*60)
print('CELL 2: DATASET, PITCH CACHE & SSL MODEL PRE-CACHE')
print('='*60)

WAV_DIR = pathlib.Path('/kaggle/input/samuel-voice-samples')
if not WAV_DIR.exists():
    print(f'[WARN] Expected dataset at {WAV_DIR}, searching /kaggle/input...')
    for p in pathlib.Path('/kaggle/input').glob('*'):
        if list(p.rglob('*.wav')):
            WAV_DIR = p
            print(f'Found wavs at {WAV_DIR}')
            break

wavs = list(WAV_DIR.rglob('*.wav')) if WAV_DIR.exists() else []
print(f'Target WAV_DIR: {WAV_DIR}')
print(f'Found {len(wavs)} WAV files.')

if len(wavs) == 0:
    print('[FATAL] No WAV files found. Ensure \'lydorandlydor/samuel-voice-samples\' is attached in Settings.')
    sys.exit(1)

# Check for cached pitch cache from previous run (persistence)
manifest_path = pathlib.Path('manifests/custom.jsonl')
cache_path = pathlib.Path('manifests/pitch_cache/custom_spf512.npz')
cached_manifest = pathlib.Path('/kaggle/input/samuel-realtime-parrot-custom-train/manifests/custom.jsonl')
cached_cache = pathlib.Path('/kaggle/input/samuel-realtime-parrot-custom-train/manifests/pitch_cache/custom_spf512.npz')
if cached_manifest.exists() and cached_cache.exists():
    wavs_mtime = max(p.stat().st_mtime for p in wavs)
    cache_mtime = cached_cache.stat().st_mtime
    if cache_mtime > wavs_mtime:
        print(f'Found cached pitch cache from previous run, restoring (cache newer than wavs)...')
        manifest_path.parent.mkdir(parents=True, exist_ok=True)
        cache_path.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(cached_manifest, manifest_path)
        shutil.copy(cached_cache, cache_path)
        print('Restored cached pitch cache (skipping 2min prepare)')
        skip_prepare = True
    else:
        print('Cached pitch cache is stale (wavs newer), will regenerate')
        skip_prepare = False
else:
    skip_prepare = False

if not skip_prepare:
    print('\n[1/2] Running prepare_custom_dataset.py...')
    cmd = f'uv run python scripts/prepare_custom_dataset.py --wav-dir {WAV_DIR} --manifest manifests/custom.jsonl --pitch-cache manifests/pitch_cache/custom_spf512.npz --sample-rate 44100 --samples-per-frame 512'
    print(f'Executing: {cmd}')
    subprocess.run(cmd, shell=True, check=True)
else:
    print('Skipped prepare_custom_dataset.py (using cached)')

print('\n[2/2] Verifying outputs...')
print(f'Manifest lines: {sum(1 for _ in open(manifest_path))}')
d = np.load(cache_path)
print(f'Pitch cache keys: {list(d.files)[:6]}...')
print(f'  n_files: {d["n_files"]}, sr: {d["sample_rate"]}, spf: {d["samples_per_frame"]}')

# --- Pre-cache SSL model (wav2vec2-base-960h) to avoid DDP download race ---
print('\n[3/3] Pre-caching SSL model (facebook/wav2vec2-base-960h)...')
pre_cache_code = '''
import os
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'
from transformers import AutoModel
print('Downloading wav2vec2-base-960h...')
model = AutoModel.from_pretrained('facebook/wav2vec2-base-960h')
print('SSL model cached successfully')
'''
result = subprocess.run(['python', '-c', pre_cache_code], capture_output=True, text=True, timeout=300)
if result.returncode != 0:
    print(f'[WARN] SSL model pre-cache failed: {result.stderr[:500]}')
    print('Will try during training (may hit rate limits)...')
else:
    print('✅ SSL model pre-cached')

print('='*60)


In [ ]:
import subprocess, os, sys, traceback
print('='*60)
print('CELL 3: DISTRIBUTED TRAINING (DDP on 2x T4)')
print('='*60)

print('[1/3] GPU Status:')
print(subprocess.getoutput('nvidia-smi'))

print('\n[2/3] Environment variables for DDP:')
os.environ['NCCL_DEBUG'] = 'INFO'
os.environ['PYTHONUNBUFFERED'] = '1'
os.environ['TORCH_NCCL_ASYNC_ERROR_HANDLING'] = '1'
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'

# HF_TOKEN for model downloads (optional, for rate limiting)
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
    os.environ['HF_TOKEN'] = HF_TOKEN
    print('[INFO] HF_TOKEN loaded from secrets')
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN', '')
    if HF_TOKEN:
        print('[INFO] HF_TOKEN loaded from env')
    else:
        print('[INFO] No HF_TOKEN - using public access (may hit rate limits)')

# Absolute paths for manifest and pitch cache (created in root, not vendor/samuel)
MANIFEST_PATH = '/kaggle/working/samuel-realtime-parrot/manifests/custom.jsonl'
PITCH_CACHE_PATH = '/kaggle/working/samuel-realtime-parrot/manifests/pitch_cache/custom_spf512.npz'

print(f'\n[3/3] Manifest: {MANIFEST_PATH}')
print(f'Pitch cache: {PITCH_CACHE_PATH}')
print(f'Exists: {os.path.exists(MANIFEST_PATH)}, {os.path.exists(PITCH_CACHE_PATH)}')

print('\n[4/4] Launching torchrun with batch_size=16...')
base_cmd = f'''torchrun --standalone --nproc_per_node=2 -m samuel.train \
    run.name=kaggle_custom_voice_ft \
    data.manifest_path={MANIFEST_PATH} \
    data.pitch_cache_path={PITCH_CACHE_PATH} \
    batch_size=16 \
    optim.max_steps=5000 \
    optim.warmup_steps=500 \
    log.eval_every=500 \
    log.ckpt_every=1000 \
    log.wandb_mode=offline \
    log.asr_whisper_size='' '''  # Disable Whisper (CUDA/ROCm issues)

print(f'Executing:\n{base_cmd}')
# Use tee to capture output to file for debugging
with open('training.log', 'w') as log_file:
    process = subprocess.Popen(base_cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in process.stdout:
        print(line, end='')
        log_file.write(line)
        log_file.flush()
    process.wait()

if process.returncode != 0:
    print(f'\n[WARN] Training failed with exit code {process.returncode}. Attempting fallback to batch_size=8...')
    print('Last 50 lines of training log:')
    with open('training.log', 'r') as f:
        lines = f.readlines()
        for line in lines[-50:]:
            print(line, end='')
    
    fallback_cmd = base_cmd.replace('batch_size=16', 'batch_size=8')
    print(f'\nExecuting fallback:\n{fallback_cmd}')
    with open('training_fallback.log', 'w') as log_file:
        process = subprocess.Popen(fallback_cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in process.stdout:
            print(line, end='')
            log_file.write(line)
            log_file.flush()
        process.wait()
    
    if process.returncode != 0:
        print(f'\n[ERROR] Training failed even with batch_size=8. Exit code: {process.returncode}')
        print('Last 50 lines of fallback log:')
        with open('training_fallback.log', 'r') as f:
            lines = f.readlines()
            for line in lines[-50:]:
                print(line, end='')
        sys.exit(1)

print('='*60)
print('✅ Training completed successfully!')


In [ ]:
import shutil
from pathlib import Path
import glob

print('='*60)
print('CELL 4: EXTRACT & DOWNLOAD CHECKPOINT')
print('='*60)

# Find run directory (Hydra creates timestamped dirs)
run_dirs = sorted(Path('.').glob('runs/kaggle_custom_voice_ft_*'), key=lambda p: p.stat().st_mtime, reverse=True)
if run_dirs:
    latest_run = run_dirs[0]
    ckpt_dir = latest_run / 'checkpoints'
else:
    ckpt_dir = Path('runs/kaggle_custom_voice_ft/checkpoints')

print(f'Looking for checkpoints in: {ckpt_dir}')
if ckpt_dir.exists():
    checkpoints = sorted(ckpt_dir.glob('*.pt'), key=lambda p: p.stat().st_mtime, reverse=True)
    for p in checkpoints:
        print(f'  {p.name} ({p.stat().st_size/1024/1024:.1f} MB)')
    
    if checkpoints:
        latest = checkpoints[0]
        dst = Path('/kaggle/working/samuel_custom_last.pt')
        shutil.copy(latest, dst)
        
        # Also copy config.json
        src_cfg = latest.parent.parent / 'config.json'
        if src_cfg.exists():
            shutil.copy(src_cfg, '/kaggle/working/custom_config.json')
        
        print(f'✅ Checkpoint ready: {dst} ({dst.stat().st_size/1024/1024:.1f} MB)')
        print(f'   Config: /kaggle/working/custom_config.json')
        
        # Also copy to repo for persistence (Save Output)
        shutil.copy(latest, 'samuel_custom_last.pt')
        if src_cfg.exists():
            shutil.copy(src_cfg, 'custom_config.json')
        print('   Also copied to ./samuel_custom_last.pt (persists if Save Output ON)')
    else:
        print('[WARN] No .pt files found in checkpoints dir')
else:
    print('[ERROR] Checkpoint directory not found')
    print('Checking runs/...')
    for p in Path('.').rglob('*.pt'):
        print(f'  Found: {p}')

print('='*60)


In [ ]:
import os, subprocess, pathlib
print('='*60)
print('CELL 5: FINALIZE (NO HF PUSH, PUBLIC REPO, PERSISTENCE)')
print('='*60)

print('Repo is now PUBLIC (https://github.com/lydorianP/samuel-realtime-parrot)')
print('No GH_TOKEN or HF_TOKEN secrets required for clone/push.')
print('Checkpoint will be in /kaggle/working/ and persisted via Save Output.')

# Verify output files exist in /kaggle/working (persistence)
import pathlib as _pathlib
ckpt = _pathlib.Path("/kaggle/working/samuel_custom_last.pt")
cfg = _pathlib.Path("/kaggle/working/custom_config.json")
if ckpt.exists():
    print(f"✅ Checkpoint in working: {ckpt} ({ckpt.stat().st_size/1024/1024:.1f} MB)")
else:
    # Try alternative location
    for p in _pathlib.Path("runs").rglob("*.pt"):
        print(f"Found checkpoint: {p} {p.stat().st_size/1024/1024:.1f} MB")
        # Copy to working for persistence
        import shutil
        shutil.copy(p, ckpt)
        print(f"Copied to {ckpt}")

if cfg.exists():
    print(f"✅ Config in working: {cfg}")
else:
    print("Config not yet in working, but should be after Cell 4")

print("\nPersistence: Ensure Kaggle UI has 'Save Output' ON (persistence to all).")
print("The checkpoint at /kaggle/working/samuel_custom_last.pt will be available via:")
print("  kaggle kernels output lydorandlydor/samuel-realtime-parrot-custom-train -p kaggle/output/")
print("And locally via: ./scripts/re_export_custom.sh kaggle/output/samuel_custom_last.pt")

# No HF push - repo is public, download via Kaggle output is sufficient
print("\nNo HF push (scrapped per Director). For manual HF push, run locally:")
print("  hf upload barbarabhb/samuel-realtime-parrot-custom kaggle/output/samuel_custom_last.pt --repo-type model")
print('='*60)
